In [2]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Permitir que el notebook importe desde la carpeta 'src'
sys.path.append(os.path.abspath('../'))

from src.models.balance import apply_balance_to_train
from src.models.classifiers import get_random_forest_classifier
from src.models.hysteresis import apply_hysteresis_logic
from src.evaluation.metrics import compute_clinical_metrics, compute_event_metrics

# =====================================================================
# 1. GENERACIÓN DE DATOS CON DESBALANCE EXTREMO (Simulación I+D)
# =====================================================================
np.random.seed(42)
num_ventanas = 10000  # Simulamos unas 1.3 horas de monitoreo continuo
num_rasgos = 36       # 3 canales * 12 rasgos de la DWT db4

# Creamos características aleatorias uniformes de fondo
X_completo = np.random.normal(loc=0.0, scale=1.0, size=(num_ventanas, num_rasgos))
y_completo = np.zeros(num_ventanas, dtype=int)

# Inyectamos crisis ietales: solo el 0.3% de las ventanas totales (~30 ventanas)
num_ventanas_crisis = int(num_ventanas * 0.003)
indices_crisis = np.sort(np.random.choice(range(2000, 8000), size=num_ventanas_crisis, replace=False))
y_completo[indices_crisis] = 1

# Hacemos que las ventanas de crisis tengan rasgos distintivos (mayor energía, menor entropía)
X_completo[indices_crisis, :] += np.random.normal(loc=1.5, scale=0.5, size=(num_ventanas_crisis, num_rasgos))

# =====================================================================
# 2. PARTICIÓN HONESTA DE DATOS (Anti-Fuga de Datos / Data Leakage)
# =====================================================================
# Separamos un 30% para test intacto antes de tocar nada del balanceo
X_train, X_test, y_train, y_test = train_test_split(
    X_completo, y_completo, test_size=0.3, random_state=42, stratify=y_completo
)

print("=== BALANCE DE CLASES ORIGINAL ===")
print(f"Train - No-crisis: {np.sum(y_train==0)} | Crisis: {np.sum(y_train==1)}")
print(f"Test  - No-crisis: {np.sum(y_test==0)} | Crisis: {np.sum(y_test==1)}\n")

# =====================================================================
# 3. BALANCEO DE CLASES (SÓLO EN EL CONJUNTO DE TRAIN)
# =====================================================================
# Llevamos la clase minoritaria al 10% con SMOTE y luego submuestreamos
X_train_bal, y_train_bal = apply_balance_to_train(
    X_train, y_train, smote_strategy=0.1, undersample_strategy=0.5, random_state=42
)

print("=== BALANCE DE CLASES DESPUÉS DE SMOTE + SUBMUESTREO (TRAIN) ===")
print(f"Train Bal - No-crisis: {np.sum(y_train_bal==0)} | Crisis: {np.sum(y_train_bal==1)}\n")

# =====================================================================
# 4. ENTRENAMIENTO Y PREDICCIÓN PROBABILÍSTICA
# =====================================================================
clf = get_random_forest_classifier(n_estimators=100, random_state=42)
clf.fit(X_train_bal, y_train_bal)

# Obtenemos las probabilidades continuas p(crisis) sobre el conjunto de TEST
probabilidades_test = clf.predict_proba(X_test)[:, 1]

# =====================================================================
# 5. POST-PROCESAMIENTO: LÓGICA DE HISTÉRESIS
# =====================================================================
# Predicción clásica usando un umbral fijo estándar (0.5)
predicciones_fijas = (probabilidades_test >= 0.5).astype(int)

# Predicción robusta exigiendo persistencia temporal (N=3 ventanas consecutivas)
predicciones_histeresis = apply_hysteresis_logic(
    probabilidades_test, N_windows=3, threshold_high=0.6, threshold_low=0.4
)

# =====================================================================
# 6. EVALUACIÓN COMPARATIVA Y MÉTRICAS CLÍNICAS
# =====================================================================
metrics_fijas = compute_clinical_metrics(y_test, predicciones_fijas, probabilidades_test)
metrics_hist = compute_clinical_metrics(y_test, predicciones_histeresis, probabilidades_test)
event_hist = compute_event_metrics(y_test, predicciones_histeresis, fs=256, step_size=128)

print("=== COMPARATIVA DE RENDIMIENTO CLÍNICO (EN TEST) ===")
print(f"Métrica                 | Umbral Fijo (0.5) | Con Histéresis Temporal (Schmitt-trigger)")
print(f"---------------------------------------------------------------------------------------")
print(f"Sensibilidad (TPR)      | {metrics_fijas['Sensibilidad (TPR)']:17.2f} | {metrics_hist['Sensibilidad (TPR)']:39.2f}")
print(f"Especificidad (TNR)     | {metrics_fijas['Especificidad (TNR)']:17.2f} | {metrics_hist['Especificidad (TNR)']:39.2f}")
print(f"Falsos Positivos (FP)   | {metrics_fijas['Falsos Positivos (FP)']:17d} | {metrics_hist['Falsos Positivos (FP)']:39d}")
print(f"AUC-ROC                 | {metrics_fijas['AUC-ROC']:17.2f} | {metrics_hist['AUC-ROC']:39.2f}")
print(f"Tasa Falsas Alarmas     |        -          | {event_hist['Tasa de Falsas Alarmas (FA/h)']:34.2f} FA/h")

=== BALANCE DE CLASES ORIGINAL ===
Train - No-crisis: 6979 | Crisis: 21
Test  - No-crisis: 2991 | Crisis: 9

=== BALANCE DE CLASES DESPUÉS DE SMOTE + SUBMUESTREO (TRAIN) ===
Train Bal - No-crisis: 1394 | Crisis: 697

=== COMPARATIVA DE RENDIMIENTO CLÍNICO (EN TEST) ===
Métrica                 | Umbral Fijo (0.5) | Con Histéresis Temporal (Schmitt-trigger)
---------------------------------------------------------------------------------------
Sensibilidad (TPR)      |              0.67 |                                    0.00
Especificidad (TNR)     |              1.00 |                                    1.00
Falsos Positivos (FP)   |                 0 |                                       0
AUC-ROC                 |              1.00 |                                    1.00
Tasa Falsas Alarmas     |        -          |                               0.00 FA/h
